# Build and evaluate logistic regression vs. XGBoost

This notebook compares four controlled pipelines on the same synthetic cohort and data splits:

1. Minimally processed logistic regression
2. Prepared logistic regression
3. Raw-feature tree-based XGBoost
4. Prepared-feature XGBoost as a preprocessing control

For Steps 2–9, give each prompt to an AI coding assistant, paste the generated code into the empty cell(s), and run the notebook in order.

## Setup

In [5]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder

from xgboost import XGBClassifier

RANDOM_SEED = 20261013
TARGET = "hospitalized_12m"
BASELINE_DAYS = 365

sns.set_theme(style="whitegrid", context="notebook")
MODEL_COLORS = {
    "Logistic: minimal": "#D4A72C",
    "Logistic: prepared": "#2563EB",
    "XGBoost: raw": "#E76F51",
    "XGBoost: prepared": "#6B7D3A",
}
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Step 1: Build the analysis table from two CSV files

In [6]:
patients = pd.read_csv("data/patients.csv", parse_dates=["index_date"])
labs = pd.read_csv("data/labs.csv", parse_dates=["lab_date"])

labs_with_index = labs.merge(
    patients[["patient_id", "index_date"]],
    on="patient_id",
    how="left",
    validate="many_to_one",
)
baseline_start = labs_with_index["index_date"] - pd.Timedelta(days=BASELINE_DAYS)
baseline_mask = (
    (labs_with_index["lab_date"] >= baseline_start)
    & (labs_with_index["lab_date"] < labs_with_index["index_date"])
)
baseline_labs = labs_with_index.loc[baseline_mask, ["patient_id", "lab_name", "lab_value"]]
assert not baseline_labs.duplicated(["patient_id", "lab_name"]).any()

baseline_labs_wide = (
    baseline_labs.pivot(index="patient_id", columns="lab_name", values="lab_value")
    .reset_index()
    .rename_axis(columns=None)
)
model_data = patients.merge(baseline_labs_wide, on="patient_id", how="left", validate="one_to_one")
assert len(model_data) == len(patients)

display(model_data.head(3))
print(f"Modeling cohort: {len(model_data):,} patients")
print(f"Outcome prevalence: {model_data[TARGET].mean():.1%}")

,patient_id,index_date,index_medication_class,age_at_index,sex,race_ethnicity,insurance_type,smoking_status,diabetes_duration_years,bmi_baseline,...,charlson_comorbidity_index,hospitalized_12m,alt,creatinine,egfr,glucose,hba1c,hdl_cholesterol,ldl_cholesterol,triglycerides
0,P000001,2022-02-26,SGLT2 inhibitor,75,Male,Asian,Commercial,Never,4.0,47.0,...,2,0,13.0,0.68,84.0,NaN,8.2,36.0,110.0,93.0
1,P000002,2022-10-11,Insulin,63,Female,Black,Commercial,Former,15.0,NaN,...,1,0,NaN,0.99,90.0,275.0,10.9,67.0,155.0,NaN
2,P000003,2022-08-26,Metformin,79,Male,White,Medicare,Never,10.5,34.6,...,2,0,45.0,0.64,79.0,173.0,8.3,53.0,132.0,65.0


Modeling cohort: 4,000 patients
Outcome prevalence: 22.4%


## Step 2: Create fixed training, validation, and test sets

**Prompt:** Using `model_data`, create `identifiers` from `patient_id` and `index_date`; create `X` without those columns or `TARGET`, and create `y` from `TARGET`. With `RANDOM_SEED` and outcome stratification, split the data 60%/20%/20% into `X_train`, `X_validation`, `X_test`, `y_train`, `y_validation`, and `y_test` by first holding out 20% for testing and then using 25% of the development set for validation. Display `split_summary` with patient counts and outcome rates. Do not preprocess before splitting. Return one code cell only.

## Step 3: Define the minimally processed logistic-regression pipeline

**Prompt:** From `X`, define `categorical_columns` from object/string/category dtypes and `numeric_columns` as the remainder. Build `minimal_logistic_preprocessor`: median-impute numeric features; most-frequent-impute and one-hot encode categorical features with unknown categories ignored; set `verbose_feature_names_out=False`. Build `minimal_logistic` using `LogisticRegression(max_iter=3000, C=1.0, solver="lbfgs")`. Do not scale, log-transform, add missingness flags, or fit the model. Return one code cell only.

## Step 4: Define the prepared logistic-regression pipeline

**Prompt:** First inspect `data_dictionary.yaml` and use its exact column names. Return two code cells. **Cell 1:** Drop `total_visits_baseline` from `X` and each split, creating `X_prepared`, `Xp_train`, `Xp_validation`, and `Xp_test`. Define `binary_columns` from all binary predictors except the outcome. Define `log_numeric_columns` from the component visit counts, diabetes duration, medication count, Charlson index, creatinine, triglycerides, and ALT. Derive `prepared_categorical_columns`, remaining `continuous_columns`, and `columns_with_missing_flags`. **Cell 2:** Build `prepared_preprocessor` with branches named `continuous`, `log_numeric`, `missing_flags`, `binary`, and `categorical`: median imputation plus `MinMaxScaler` for continuous features; median imputation, `FunctionTransformer(np.log1p, feature_names_out="one-to-one")`, and scaling for log-numeric features; `MissingIndicator(features="missing-only", error_on_new=False)`; most-frequent imputation for binary features; and most-frequent imputation plus one-hot encoding for categorical features. Set `verbose_feature_names_out=True`, then build `prepared_logistic` with `LogisticRegression(max_iter=1500, C=1.0, solver="lbfgs")`. Do not fit it.

## Step 5: Define raw and prepared XGBoost pipelines

**Prompt:** Return two code cells. **Cell 1:** Build `raw_xgb_preprocessor` with branches named `numeric` and `categorical`: pass numeric features through unchanged; most-frequent-impute and one-hot encode categorical features; set `sparse_threshold=1.0` and `verbose_feature_names_out=False`. Define XGBoost settings (`binary:logistic`, `logloss`, `hist`, `RANDOM_SEED`, `n_jobs=1`), then create `raw_xgboost` and `prepared_xgboost`, cloning `prepared_preprocessor` for the latter. **Cell 2:** Define shuffled five-fold stratified CV and a 20-iteration AUROC `RandomizedSearchCV` with refitting, `RANDOM_SEED`, `n_jobs=2`, and no train scores. Search `n_estimators` [200,350,500,700], `learning_rate` [.02,.04,.07,.10], `max_depth` [2,3,4,5], `min_child_weight` [1,3,5,10], `subsample` [.70,.85,1], `colsample_bytree` [.70,.85,1], `gamma` [0,.10,.25], `reg_alpha` [0,.10,.50], and `reg_lambda` [1,2,5]. Create `raw_xgboost_search` and `prepared_xgboost_search`; do not fit them.

## Step 6: Train every model on the same training patients

**Prompt:** Create `models` mapping the four exact labels `Logistic: minimal`, `Logistic: prepared`, `XGBoost: raw`, and `XGBoost: prepared` to each model and its matching train/validation/test feature sets. Fit every model using `y_train`; suppress only logistic `lbfgs failed to converge` warnings. Store fitted pipelines in `fitted_models`; for randomized searches store `best_estimator_` and record `best_cross_validated_AUROC` plus `best_params_` in `xgb_tuning_results`. Display the tuning-results table. Return one code cell only.

## Step 7: Select probability thresholds on validation data

**Prompt:** Define `select_youden_threshold` using `roc_curve`, excluding nonfinite thresholds and maximizing sensitivity minus false-positive rate. For every fitted model, calculate validation and test positive-class probabilities using its matching feature set. Store them in `validation_probabilities` and `test_probabilities`, select thresholds from `y_validation` only, and store them in `selected_thresholds`. Display the thresholds to three decimals. Do not use `y_test` for threshold selection. Return one code cell only.

## Step 8: Evaluate final performance on the untouched test set

**Prompt:** Return three code cells and do not add calibration analysis. **Cell 1:** Define `classification_metrics` and build `evaluation` from `y_test`, `test_probabilities`, and validation-selected thresholds, reporting AUROC, PR-AUC, threshold, sensitivity, specificity, PPV, and NPV; display formatted results. **Cell 2:** Plot side-by-side test ROC and precision-recall curves for all models using `MODEL_COLORS`, AUROC/PR-AUC in legends, a no-discrimination diagonal, and outcome-prevalence reference line. **Cell 3:** Reshape sensitivity, specificity, PPV, and NPV and create the grouped bar chart titled `Test performance at validation-selected thresholds`, with a 0–100% vertical scale and the legend outside the plot.

## Step 9: Interpret the prepared XGBoost model with SHAP

**Prompt:** Return three code cells. **Cell 1:** Extract the fitted prepared XGBoost preprocessor and model, obtain and clean transformed feature names, transform all `Xp_test` rows into `shap_test_data`, and calculate `shap_values` with `shap.TreeExplainer(..., check_additivity=False)`; print the patient count. **Cell 2:** Create a SHAP beeswarm summary for the top 15 features, size 10×7, titled `Prepared XGBoost: SHAP value summary`. **Cell 3:** Create a top-15 mean absolute SHAP bar chart in the prepared-XGBoost color, size 10×7, titled `Prepared XGBoost: mean absolute SHAP values`. Do not refit or resample the model.